<a href="https://colab.research.google.com/github/KamilAkarsu/DoktoraDeneyleri-FracAtlas/blob/main/Exp_2_5_1_FracAtlas_and_Traditional_CNN_(vgg19_unfrozen).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Hücre 1: Kütüphane İçe Aktarımları ve Veri Ortamının Hazırlanması

# ==============================================================================
# TEMEL KÜTÜPHANELERİN İÇE AKTARILMASI
# ==============================================================================
import os           # İşletim sistemi işlemleri, klasör/dosya yolu yönetimi
import time         # Eğitim sürelerinin ve çıkarım (inference) hızlarının hesaplanması
import zipfile      # Sıkıştırılmış veri setinin (FracAtlas) yerel diske çıkartılması

# ==============================================================================
# PYTORCH VE DERİN ÖĞRENME MODÜLLERİ
# ==============================================================================
import torch        # Ana derin öğrenme çerçevesi
import torch.nn as nn # Sinir ağı katmanları ve kayıp (loss) fonksiyonları
import torch.optim as optim # Optimizasyon algoritmaları (AdamW vb. ve Öğrenme Oranı Planlayıcıları)
from torch.utils.data import Dataset, DataLoader # Özelleştirilmiş veri seti ve toplu veri yükleyici sınıfları
from torchvision import transforms, models # Dinamik veri artırımı (augmentation) ve ön-eğitimli jenerik modeller

# ==============================================================================
# VERİ İŞLEME VE GÖRSELLEŞTİRME MODÜLLERİ
# ==============================================================================
from PIL import Image # Tıbbi görüntülerin (Röntgen) diskten okunması
import numpy as np  # Vektörel matris işlemleri
import pandas as pd # Eğitim geçmişinin ve sonuçların tablo halinde kaydedilmesi (CSV)
from tqdm import tqdm # Eğitim ve doğrulama döngülerinde iterasyon ilerleyişinin görselleştirilmesi

# ==============================================================================
# PERFORMANS METRİKLERİ (SCIKIT-LEARN)
# ==============================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             mean_absolute_error, mean_squared_error)

# ==============================================================================
# ÇALIŞMA ORTAMI VE VERİ BAZI HAZIRLIKLARI
# ==============================================================================
# 1. Google Drive Bağlantısı: Ağırlıkların ve grafiklerin kaydedileceği kalıcı disk
from google.colab import drive
drive.mount('/content/drive')

# 2. Veri Setinin Yerel Diske Çıkartılması
# Bulut depolamadan (Drive) doğrudan veri okumak I/O darboğazına (bottleneck) yol açar.
# GPU'nun veriyi beklemesini engellemek için veri seti geçici, hızlı yerel belleğe alınır.
DRIVE_ZIP_YOLU = '/content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_FracAtlas.zip'
YEREL_VERI_KLASORU = '/content/dataset'

if not os.path.exists(YEREL_VERI_KLASORU):
    print(f"Veri seti yerel diske çıkartılıyor: {DRIVE_ZIP_YOLU} ...")
    with zipfile.ZipFile(DRIVE_ZIP_YOLU, 'r') as zip_ref:
        zip_ref.extractall(YEREL_VERI_KLASORU)
    print("Çıkartma işlemi tamamlandı.")
else:
    print("Veri seti yerel diskte zaten mevcut.")

# ==============================================================================
# DONANIM HIZLANDIRICI ATAMASI
# ==============================================================================
# A100 GPU Kontrolü: Tensor işlemlerinin CPU yerine CUDA mimarisinde yürütülmesi için
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif İşlem Birimi: {device}")

Mounted at /content/drive
Veri seti yerel diske çıkartılıyor: /content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_FracAtlas.zip ...
Çıkartma işlemi tamamlandı.
Aktif İşlem Birimi: cuda


hücre 1 sözde kod

In [2]:
# ==============================================================================
# HÜCRE 2: İZOLE HİPERPARAMETRE KONFİGÜRASYONU (EXP 2.5.1 - FRACATLAS VGG19 UNFROZEN)
# ==============================================================================

CONFIG = {
    "experiment_name": "Exp 2.5.1: FracAtlas and Traditional CNN (vgg19_unfrozen)",

    # ⚠️ MİMARİ VGG19 OLARAK DEĞİŞTİRİLDİ
    "model_architecture": "vgg19",

    "freeze_backbone": False,

    "target_image_size": (224, 224),
    "batch_size": 32,
    "learning_rate": 5e-5,
    "num_epochs": 50,
    "weight_decay": 5e-3,
    "early_stopping_patience": 12,
    "use_mixup": False,
    "mixup_alpha": 0.2,
    "num_workers": 4,

    "augmentations": {
        "random_rotation_degrees": 15,
        "horizontal_flip_prob": 0.5,
        "color_jitter_brightness": 0.2,
        "color_jitter_contrast": 0.2
    }
}

print(f"✅ {CONFIG['experiment_name']} konfigürasyonu başarıyla yüklendi.")
print(f"🔥 Strateji: UNFROZEN (Tüm ağ ağırlıkları güncellenecek) 🔥")

✅ Exp 2.5.1: FracAtlas and Traditional CNN (vgg19_unfrozen) konfigürasyonu başarıyla yüklendi.
🔥 Strateji: UNFROZEN (Tüm ağ ağırlıkları güncellenecek) 🔥


hücre 2 sözde kod

In [3]:
#Hücre 3: Jenerik Veri Yükleyici ve Dinamik Artırma

import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

# ==============================================================================
# ÖZELLEŞTİRİLMİŞ TIBBİ GÖRÜNTÜ VERİ KÜMESİ SINIFI (FRACATLAS)
# ==============================================================================
class JenerikMedikalDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # FracAtlas etiketleri klasör isimlerinde gizlidir:
        # 'fractured' = 1 (Kırık), 'non_fractured' = 0 (Sağlam)
        for root, dirs, files in os.walk(root_dir):
            for img_name in files:
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    tam_yol = os.path.join(root, img_name)
                    klasor_yolu_kucuk = tam_yol.lower()

                    # Sınıflandırma problemi için etiket çıkarma (Label Extraction)
                    # 'non_fractured' kontrolü önce yapılmalıdır, aksi halde 'fractured' ikisiyle de eşleşir.
                    if 'non_fractured' in klasor_yolu_kucuk:
                        label = 0
                    elif 'fractured' in klasor_yolu_kucuk:
                        label = 1
                    else:
                        continue # Alakasız klasörleri atla

                    self.image_paths.append(tam_yol)
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

# =====================================================================
# DÖNÜŞÜMLER (TRANSFORMS) VE VERİ ARTIRMA (AUGMENTATION)
# =====================================================================
# Doğrulama setine sadece boyutlandırma ve normalizasyon uygulanır.
base_transforms = transforms.Compose([
    transforms.Resize(CONFIG["target_image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Eğitim setine, aşırı öğrenmeyi önlemek için izole CONFIG dosyasındaki deformasyonlar uygulanır.
train_transforms = transforms.Compose([
    transforms.Resize(CONFIG["target_image_size"]),
    transforms.RandomRotation(CONFIG["augmentations"]["random_rotation_degrees"]),
    transforms.RandomHorizontalFlip(p=CONFIG["augmentations"]["horizontal_flip_prob"]),
    transforms.ColorJitter(
        brightness=CONFIG["augmentations"]["color_jitter_brightness"],
        contrast=CONFIG["augmentations"]["color_jitter_contrast"]
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# =====================================================================
# DİNAMİK VERİ BÖLME (TRAIN/VAL SPLIT) VE DATALOADER
# =====================================================================
# Tüm veri setini hafızaya almadan yollarını tararız
full_dataset = JenerikMedikalDataset(root_dir=YEREL_VERI_KLASORU, transform=None)

# %80 Eğitim, %20 Doğrulama (Validation) Ayrımı
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

# Seed (42) sabitleyerek her çalıştırmada aynı resimlerin aynı setlere düşmesini garantiliyoruz
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=generator)

# Alt kümeler (Subsets) üzerinde farklı dönüşümler uygulayabilmek için sarmalayıcı (Wrapper) sınıf
class TransformWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        image_path = self.subset.dataset.image_paths[self.subset.indices[index]]
        label = self.subset.dataset.labels[self.subset.indices[index]]
        image = Image.open(image_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.subset)

# Ayrı ayrı dönüşümleri (Augmentation vs Base) uygula
train_dataset = TransformWrapper(train_subset, train_transforms)
val_dataset = TransformWrapper(val_subset, base_transforms)

# DataLoader nesneleri oluşturma
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Toplam Görüntü: {len(full_dataset)} | Eğitim Verisi: {len(train_dataset)} | Doğrulama Verisi: {len(val_dataset)}")

# ==============================================================================
# MIXUP REGÜLARİZASYONU (İSTEĞE BAĞLI / ABLASYON)
# ==============================================================================
def mixup_data(x, y, alpha=CONFIG["mixup_alpha"]):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

Toplam Görüntü: 4083 | Eğitim Verisi: 3267 | Doğrulama Verisi: 816


hücre 3 sözde kod

In [5]:
# ==============================================================================
# HÜCRE 4: KAPSAMLI JENERİK MODEL OLUŞTURUCU (UNFROZEN / YENİDEN REVİZE)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

def jenerik_model_olustur(model_adi, num_classes=2, dropout_rate=0.5):
    print(f"[{model_adi}] mimarisi ImageNet ağırlıklarıyla yükleniyor... (Dropout: {dropout_rate})")

    # --- VGG16 ---
    if model_adi == "vgg19":
        model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier[6].in_features, num_classes)
        )

    # --- ALEXNET (Legacy Baseline) ---
    elif model_adi == "alexnet":
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[6].in_features, num_classes))

    # --- RESNET & RESNEXT AİLESİ ---
    elif model_adi == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))
    elif model_adi == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))
    elif model_adi == "resnext50_32x4d":
        model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))

    # --- DENSENET AİLESİ ---
    elif model_adi == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier.in_features, num_classes))
    elif model_adi == "densenet169":
        model = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier.in_features, num_classes))
    elif model_adi == "densenet201":
        model = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier.in_features, num_classes))

    # --- EFFICIENTNET AİLESİ ---
    elif model_adi == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[1].in_features, num_classes))
    elif model_adi == "efficientnet_b4":
        model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[1].in_features, num_classes))
    elif model_adi == "efficientnet_v2_s":
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[1].in_features, num_classes))

    # --- CONVNEXT AİLESİ ---
    elif model_adi == "convnext_base":
        model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        model.classifier[2] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[2].in_features, num_classes))

    # --- REGNET AİLESİ ---
    elif model_adi == "regnet_y_3_2gf":
        model = models.regnet_y_3_2gf(weights=models.RegNet_Y_3_2GF_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))

    # --- MOBILENET ---
    elif model_adi == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[3] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[3].in_features, num_classes))

    # ==========================================================
    # --- VISION TRANSFORMERS (ViT) ---
    # ==========================================================
    elif model_adi == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads.head = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.heads.head.in_features, num_classes))

    else:
        raise ValueError(f"HATA: '{model_adi}' tanımlı bir model değil. Lütfen CONFIG sözlüğünü kontrol edin.")

    # ==========================================================
    # ⚠️ DİNAMİK STRATEJİ (UNFROZEN / FROZEN) YÖNETİMİ ⚠️
    # ==========================================================
    dondurulan_katman_sayisi = 0
    acik_katman_sayisi = 0

    # Hücre 2'den gelen 'freeze_backbone' komutuna göre davranır
    is_frozen = CONFIG.get("freeze_backbone", True)

    if not is_frozen:
        # UNFROZEN SENARYOSU (Tüm ağ eğitilir)
        for param in model.parameters():
            param.requires_grad = True
            acik_katman_sayisi += 1
        print(f"Strateji: UNFROZEN. Ağdaki tüm ({acik_katman_sayisi}) katmanlar FracAtlas'a uyarlanacak.")
    else:
        # FROZEN SENARYOSU (Orijinal Kod - Sadece belirli katmanlar eğitilir)
        trainable_keywords = [
            "layer3", "layer4", "denseblock4", "features.7", "features.8",
            "features.15", "features.16", "trunk_output.block4",
            "encoder_layer_11", "heads", "fc", "classifier", "head"
        ]
        for name, param in model.named_parameters():
            if any(keyword in name for keyword in trainable_keywords):
                param.requires_grad = True
                acik_katman_sayisi += 1
            else:
                param.requires_grad = False
                dondurulan_katman_sayisi += 1
        print(f"Strateji: FROZEN. {dondurulan_katman_sayisi} katman donduruldu, {acik_katman_sayisi} katman eğitiliyor.")

    return model.to(device)

# Modeli başlatma
model = jenerik_model_olustur(CONFIG["model_architecture"], dropout_rate=0.5)

# ==========================================================
# FARKILAŞTIRILMIŞ ÖĞRENME ORANI (DISCRIMINATIVE FINE-TUNING)
# ==========================================================
head_params = []
backbone_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    # Sınıflandırma katmanlarını (Head) ayır
    if any(keyword in name for keyword in ["fc", "classifier", "heads", "head"]):
        head_params.append(param)
    else:
        # Geri kalan her şey (Unfrozen senaryosunda tüm backbone buraya düşer)
        backbone_params.append(param)

# Optimizer: Backbone (1/10 LR) ve Head (Normal LR)
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CONFIG["learning_rate"] / 10},
    {'params': head_params, 'lr': CONFIG["learning_rate"]}
], weight_decay=CONFIG["weight_decay"])

# ==========================================================
# AĞIRLIKLI KAYIP FONKSİYONU (CLASS IMBALANCE HANDLING)
# ==========================================================
class_weights = torch.tensor([1.0, 1.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Model başarıyla GPU'ya ({device}) taşındı ve eğitime hazır.")

[vgg19] mimarisi ImageNet ağırlıklarıyla yükleniyor... (Dropout: 0.5)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 237MB/s]


Strateji: UNFROZEN. Ağdaki tüm (38) katmanlar FracAtlas'a uyarlanacak.
Model başarıyla GPU'ya (cuda) taşındı ve eğitime hazır.


hücre 4 sözde kod

In [6]:
#Hücre 5: Kapsamlı Başarım Metrikleri Hesaplayıcısı

# ==============================================================================
# KAPSAMLI BAŞARIM METRİKLERİ HESAPLAYICISI
# ==============================================================================
# Modelin ürettiği tahminleri (predictions) ve olasılık skorlarını (probabilities)
# gerçek etiketlerle (ground truth) karşılaştırarak, tıbbi tanı sistemleri için
# literatürde kabul gören tüm değerlendirme ölçütlerini tek seferde hesaplar.

def kapsamli_metrikleri_hesapla(y_true, y_pred, y_scores):
    # Karmaşıklık Matrisi (Confusion Matrix) hesaplaması
    # tn (True Negative), fp (False Positive), fn (False Negative), tp (True Positive)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)

    # Scikit-learn fonksiyonları kullanılarak metriklerin sözlük yapısında toplanması
    metrikler = {
        # Genel doğruluk oranı (Sınıf dengesizliği olan durumlarda tek başına yanıltıcı olabilir)
        "Accuracy": accuracy_score(y_true, y_pred),

        # Kesinlik: Modelin "Kırık (1)" dediklerinin ne kadarının gerçekten kırık olduğu
        "Precision": precision_score(y_true, y_pred, zero_division=0),

        # Duyarlılık (Recall/Sensitivity): Gerçekte "Kırık" olanların ne kadarının model tarafından bulunabildiği
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),

        # Özgüllük (Specificity): Gerçekte "Sağlıklı" olanların ne kadarının doğru tespit edildiği
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,

        # F1-Skoru: Kesinlik ve Duyarlılık metriklerinin harmonik ortalaması
        "F1_Measure": f1_score(y_true, y_pred, zero_division=0),

        # Cohen's Kappa: Şans eseri oluşan doğru tahminleri cezalandıran, Stanford'ın MURA
        # yarışmasında ana sıralama (leaderboard) ölçütü olarak kullandığı en kritik metrik.
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),

        # ROC-AUC: Alıcı İşletim Karakteristiği Eğrisi Altında Kalan Alan (Modelin sınıfları ayırma gücü)
        "ROC_AUC": roc_auc_score(y_true, y_scores),

        # PR-AUC (uAP): Kesinlik-Duyarlılık Eğrisi Altında Kalan Alan. Dengesiz veri setlerinde
        # ROC-AUC'ye kıyasla model başarısını çok daha objektif yansıtır.
        "PR_AUC_uAP": average_precision_score(y_true, y_scores), # Uninterpolated Average Precision

        # Hata Metrikleri: Tahmin edilen olasılıklar ile gerçek etiketler (0 veya 1) arasındaki mutlak ve karesel sapmalar
        "MAE": mean_absolute_error(y_true, y_scores),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_scores))
    }
    return metrikler

hücre 5 sözde kod

In [8]:
import time
import os
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

# ==============================================================================
# 1. DOSYA YOLLARI VE KLASÖR YAPILANDIRMASI (Drive Odaklı)
# ==============================================================================
# Çıktıların kaydedileceği ana dizin (Drive üzerinde)
deney_ana_dizini = f"/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/FracAtlas Deneyleri/{CONFIG['experiment_name']}_Sonuclar"
os.makedirs(deney_ana_dizini, exist_ok=True)

# Dosya isimleri için model ön eki (Prefix) tanımlama
prefix = CONFIG['model_architecture']
model_save_path = os.path.join(deney_ana_dizini, f"{prefix}_unfrozen_best_model.pth")
csv_save_path = os.path.join(deney_ana_dizini, f"{prefix}_unfrozen_Egitim_Metrikleri.csv")
json_save_path = os.path.join(deney_ana_dizini, f"{prefix}_unfrozen_Hiperparametreler.json")

# ==============================================================================
# EĞİTİM DÖNGÜSÜ HAZIRLIKLARI
# ==============================================================================
best_val_loss = float('inf')
patience_counter = 0
egitim_gecmisi = []

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"[{CONFIG['experiment_name']}] Eğitim Başlıyor...")
print(f"🚀 En iyi model ve loglar '{prefix}' ön ekiyle kaydedilecek: {deney_ana_dizini}")
toplam_baslangic_zamani = time.time()

# ==============================================================================
# ANA EPOK DÖNGÜSÜ
# ==============================================================================
for epoch in range(CONFIG["num_epochs"]):
    epoch_baslangic_zamani = time.time()

    # --- 1. EĞİTİM FAZI ---
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} - Eğitim")
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        if CONFIG.get("use_mixup", False):
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels)
            outputs = model(inputs)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        loop.set_postfix(loss=loss.item())

    epoch_train_loss = train_loss / len(train_loader.dataset)

    # --- 2. DOĞRULAMA FAZI ---
    model.eval()
    val_loss = 0.0
    y_true, y_pred, y_scores = [], [], []
    toplam_cikarim_suresi = 0.0
    ornek_sayisi = 0

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Doğrulama"):
            inputs, labels = inputs.to(device), labels.to(device)
            start_infer = time.time()
            outputs = model(inputs)
            end_infer = time.time()
            toplam_cikarim_suresi += (end_infer - start_infer)
            ornek_sayisi += inputs.size(0)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_scores.extend(probs[:, 1].cpu().numpy())

    epoch_val_loss = val_loss / len(val_loader.dataset)
    ms_per_step = (toplam_cikarim_suresi / ornek_sayisi) * 1000
    scheduler.step(epoch_val_loss)
    metrikler = kapsamli_metrikleri_hesapla(y_true, y_pred, y_scores)

    epoch_suresi_sn = time.time() - epoch_baslangic_zamani
    current_lr = optimizer.param_groups[-1]['lr']

    print(f"\n--- Epoch {epoch+1} Sonuçları ---")
    print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Süre: {epoch_suresi_sn:.2f} sn | LR: {current_lr:.6f}")
    print(f"Accuracy: {metrikler['Accuracy']:.4f} | F1-Measure: {metrikler['F1_Measure']:.4f} | Kappa: {metrikler['Cohen_Kappa']:.4f}")
    print(f"PR-AUC (uAP): {metrikler['PR_AUC_uAP']:.4f} | ROC-AUC: {metrikler['ROC_AUC']:.4f}")
    print(f"Specificity: {metrikler['Specificity']:.4f} | Inference Time: {ms_per_step:.2f} ms/image")

    # Metrikleri listeye ekle
    metrikler.update({
        "Epoch": epoch+1,
        "Train_Loss": epoch_train_loss,
        "Val_Loss": epoch_val_loss,
        "Inference_Time_ms": ms_per_step,
        "Epoch_Suresi_sn": epoch_suresi_sn,
        "Learning_Rate": current_lr
    })
    egitim_gecmisi.append(metrikler)

    # ==========================================================================
    # MODEL KAYDETME (Doğrudan Drive'a)
    # ==========================================================================
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), model_save_path) # Kalıcı Drive kaydı
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"\nErken Durdurma tetiklendi!")
            break

# ==============================================================================
# SONUÇLARI DIŞA AKTARMA (Prefixli İsimler)
# ==============================================================================
# TOPLAM SÜREYİ HESAPLAMAK İÇİN BURASI GEREKLİ
toplam_bitis_zamani = time.time()
toplam_sure_dk = (toplam_bitis_zamani - toplam_baslangic_zamani) / 60
print(f"\nToplam Eğitim Süresi: {toplam_sure_dk:.2f} dakika.")

# 1. CSV Kaydı
pd.DataFrame(egitim_gecmisi).to_csv(csv_save_path, index=False)

# 2. JSON Kaydı (Süre ve LR geçmişi eklendi)
kayit_verisi = CONFIG.copy()
kayit_verisi["Zaman_Bilgileri"] = {
    "Toplam_Egitim_Suresi_Dakika": round(toplam_sure_dk, 2),
    "Epoch_Sureleri_Saniye": [round(m["Epoch_Suresi_sn"], 2) for m in egitim_gecmisi]
}
kayit_verisi["Learning_Rate_Gecmisi"] = [m["Learning_Rate"] for m in egitim_gecmisi]

with open(json_save_path, "w", encoding="utf-8") as f:
    json.dump(kayit_verisi, f, indent=4, ensure_ascii=False)


# 3. Grafiklerin Kaydı (Prefixli)
# Loss Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(pd.DataFrame(egitim_gecmisi)['Epoch'], pd.DataFrame(egitim_gecmisi)['Train_Loss'], label='Train Loss')
plt.plot(pd.DataFrame(egitim_gecmisi)['Epoch'], pd.DataFrame(egitim_gecmisi)['Val_Loss'], label='Val Loss')
plt.title(f'{prefix} - Training and Validation Loss')
plt.legend()
plt.savefig(os.path.join(deney_ana_dizini, f"{prefix}_1_Loss_Curve.png"), dpi=300)
plt.close()

# Accuracy Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(pd.DataFrame(egitim_gecmisi)['Epoch'], pd.DataFrame(egitim_gecmisi)['Accuracy'], label='Accuracy', color='green')
plt.title(f'{prefix} - Accuracy')
plt.legend()
plt.savefig(os.path.join(deney_ana_dizini, f"{prefix}_2_Accuracy_Curve.png"), dpi=300)
plt.close()

# ==============================================================================
# EN İYİ MODELİ GERİ YÜKLEME VE NİHAİ GRAFİKLER
# ==============================================================================
print(f"\nEn İyi Model ({prefix}) ağırlıkları geri yükleniyor...")
model.load_state_dict(torch.load(model_save_path))
model.eval()

y_true_best, y_pred_best, y_scores_best = [], [], []
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Best Model Değerlendirmesi"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        y_true_best.extend(labels.cpu().numpy())
        y_pred_best.extend(preds.cpu().numpy())
        y_scores_best.extend(probs[:, 1].cpu().numpy())


# 3. Karmaşıklık Matrisi (Heatmap)
cm_default = confusion_matrix(y_true_best, y_pred_best)
tn, fp, fn, tp = cm_default.ravel() if cm_default.size == 4 else (0, 0, 0, 0)
cm_custom = np.array([[tp, fp],
                      [fn, tn]])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Wistia', cbar=False,
            annot_kws={"size": 16, "weight": "bold"},
            xticklabels=['Actual Positive (1)', 'Actual Negative (0)'],
            yticklabels=['Predicted Positive (1)', 'Predicted Negative (0)'],
            linewidths=1, linecolor='black')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Actual Values', fontsize=14, fontweight='bold', labelpad=10)
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold', labelpad=10)
plt.yticks(rotation=90, va="center")
plt.tight_layout()
plt.savefig(os.path.join(deney_ana_dizini, f"{prefix}_3_Confusion_Matrix.png"), dpi=300)
plt.close()

# ROC Eğrisi
fpr, tpr, _ = roc_curve(y_true_best, y_scores_best)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title(f'{prefix} - ROC Curve')
plt.legend()
plt.savefig(os.path.join(deney_ana_dizini, f"{prefix}_4_ROC_Curve.png"), dpi=300)
plt.close()

print(f"\n✅ Tüm sonuçlar '{prefix}' ön ekiyle '{deney_ana_dizini}' klasörüne kaydedildi.")

[Exp 2.5.1: FracAtlas and Traditional CNN (vgg19_unfrozen)] Eğitim Başlıyor...
🚀 En iyi model ve loglar 'vgg19' ön ekiyle kaydedilecek: /content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/FracAtlas Deneyleri/Exp 2.5.1: FracAtlas and Traditional CNN (vgg19_unfrozen)_Sonuclar


Doğrulama: 100%|██████████| 26/26 [00:00<00:00, 33.24it/s]



--- Epoch 1 Sonuçları ---
Train Loss: 0.0754 | Val Loss: 0.6737 | Süre: 8.33 sn | LR: 0.000006
Accuracy: 0.8909 | F1-Measure: 0.6716 | Kappa: 0.6068
PR-AUC (uAP): 0.7213 | ROC-AUC: 0.8567
Specificity: 0.9507 | Inference Time: 0.11 ms/image


Doğrulama: 100%|██████████| 26/26 [00:00<00:00, 32.26it/s]



--- Epoch 2 Sonuçları ---
Train Loss: 0.0769 | Val Loss: 0.6996 | Süre: 8.31 sn | LR: 0.000006
Accuracy: 0.8922 | F1-Measure: 0.6667 | Kappa: 0.6033
PR-AUC (uAP): 0.7178 | ROC-AUC: 0.8540
Specificity: 0.9567 | Inference Time: 0.10 ms/image


...


--- Epoch 12 Sonuçları ---
Train Loss: 0.0563 | Val Loss: 0.7972 | Süre: 8.35 sn | LR: 0.000002
Accuracy: 0.8934 | F1-Measure: 0.6667 | Kappa: 0.6044
PR-AUC (uAP): 0.7188 | ROC-AUC: 0.8576
Specificity: 0.9596 | Inference Time: 0.12 ms/image


Doğrulama: 100%|██████████| 26/26 [00:00<00:00, 32.95it/s]



--- Epoch 13 Sonuçları ---
Train Loss: 0.0529 | Val Loss: 0.8174 | Süre: 8.27 sn | LR: 0.000001
Accuracy: 0.8946 | F1-Measure: 0.6641 | Kappa: 0.6032
PR-AUC (uAP): 0.7195 | ROC-AUC: 0.8580
Specificity: 0.9641 | Inference Time: 0.10 ms/image

Erken Durdurma tetiklendi!

Toplam Eğitim Süresi: 1.82 dakika.

En İyi Model (vgg19) ağırlıkları geri yükleniyor...


Best Model Değerlendirmesi: 100%|██████████| 26/26 [00:00<00:00, 33.40it/s]



✅ Tüm sonuçlar 'vgg19' ön ekiyle '/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/FracAtlas Deneyleri/Exp 2.5.1: FracAtlas and Traditional CNN (vgg19_unfrozen)_Sonuclar' klasörüne kaydedildi.


hücre 6 sözde kod

In [9]:
from IPython.display import HTML

HTML("""
<audio id="audio1" src="https://www.soundjay.com/door_c2026/sounds/doorbell-5.mp3"></audio>
<audio id="audio2" src="https://www.soundjay.com/door_c2026/sounds/doorbell-6.mp3"></audio>
<audio id="audio3" src="https://www.soundjay.com/door_c2026/sounds/doorbell-7.mp3"></audio>

<script>
const a1 = document.getElementById('audio1');
const a2 = document.getElementById('audio2');
const a3 = document.getElementById('audio3');

a1.onended = () => a2.play();
a2.onended = () => a3.play();

// ilkini başlat
a1.play();
</script>
""")